In [1]:
# ============================================
# JUPYTER NOTEBOOK - GraphRAG con Ollama + RDF
# ============================================

# ============================================
# CELDA 1 - IMPORTS
# ============================================

import os
import json
import re
import datetime
import asyncio


from time import time, sleep
from uuid import uuid4
from openai import OpenAI, AsyncOpenAI
from rdflib import Graph

# Imports propios
from searchInGraph import (
    buscar_frecuentes_por_opcion,
    inferir_valor_adecuado
)

from formatHelper import (
    extraer_support_category,
    extraer_cliente,
    formatear_para_llm,
    arreglar_lista_llm,
    #merge_listas_or,
    extraer_respuesta_limpia_llm,
    merge_lista_y_parametro,
    formatear_datos_existentes_LLM
    
)

import config

import nest_asyncio
nest_asyncio.apply()

In [2]:
# ============================================
# CELDA 2 - CARGA DEL GRAFO RDF
# ============================================

graph = Graph()

graph.parse(
    config.TTL_FILE_PATH,
    format=config.TTL_FORMAT
)

print("Grafo cargado correctamente")
print(f"Número de triples: {len(graph)}")

Grafo cargado correctamente
Número de triples: 513651


In [3]:
# ============================================
# CELDA 3 - CONFIGURACIÓN DEL MODELO
# ============================================

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

mi_modelo = "mistral:latest"

print(f"Modelo configurado: {mi_modelo}")

Modelo configurado: mistral:latest


In [4]:
# ============================================
# CELDA 4 - FUNCIONES AUXILIARES
# ============================================

def open_file(filepath):
    with open(filepath, 'r', encoding='utf-8') as infile:
        return infile.read()


def save_file(filepath, content):
    with open(filepath, 'w', encoding='utf-8') as outfile:
        outfile.write(content)


def load_json(filepath):
    with open(filepath, 'r', encoding='utf-8') as infile:
        return json.load(infile)


def save_json(filepath, payload):
    with open(filepath, 'w', encoding='utf-8') as outfile:
        json.dump(
            payload,
            outfile,
            ensure_ascii=False,
            sort_keys=True,
            indent=2
        )


def timestamp_to_datetime(unix_time):
    return datetime.datetime.fromtimestamp(
        unix_time
    ).strftime("%A, %B %d, %Y at %I:%M%p %Z")

In [5]:
# ============================================
# CELDA 5 - FUNCIÓN DE COMPLETADO LLM
# ============================================

def text_completion(prompt, engine=config.MI_MODELO):

    max_retry = 5
    retry = 0

    while True:

        try:

            response = client.chat.completions.create(
                messages=[
                    {
                        "role": "user",
                        "content": prompt,
                    }
                ],
                model=engine
            )

            text = response.choices[0].message.content

            # Limpieza básica
            text = re.sub(r'[\r\n]+', '\n', text)
            text = re.sub(r'[\t ]+', ' ', text)

            return text

        except Exception as oops:

            retry += 1

            if retry >= max_retry:
                return f"Model error: {oops}"

            print("Error comunicando con el modelo:", oops)

            sleep(config.RETRY_DELAY_SECONDS)
            
            
            
async def text_completion_async(prompt, engine=config.MI_MODELO):
    """Procesa un solo prompt de forma asíncrona con manejo de reintentos."""
    max_retry = 5
    retry = 0

    while True:
        try:
            response = await config.async_client.chat.completions.create(
                messages=[
                    {
                        "role": "user",
                        "content": prompt,
                    }
                ],
                model=engine
            )

            text = response.choices[0].message.content

            # Limpieza básica
            text = re.sub(r'[\r\n]+', '\n', text)
            text = re.sub(r'[\t ]+', ' ', text)

            return text

        except Exception as oops:
            retry += 1
            if retry >= max_retry:
                return f"Model error: {oops}"

            print(f"Error comunicando con el modelo (Intento {retry}/{max_retry}):", oops)
            # Usamos asyncio.sleep en lugar de time.sleep para no bloquear el hilo
            await asyncio.sleep(config.RETRY_DELAY_SECONDS)

# --- FUNCIÓN PRINCIPAL PARA PROCESAR EL BATCH ---
async def text_completion_batch(prompts_list, engine=config.MI_MODELO):
    """
    Recibe una lista de prompts, los ejecuta en paralelo 
    y devuelve una lista con las respuestas en el mismo orden.
    """
    # Creamos una tarea asíncrona por cada prompt en la lista
    tasks = [text_completion_async(prompt, engine) for prompt in prompts_list]
    
    # Ejecutamos todas las tareas en paralelo y esperamos los resultados
    results = await asyncio.gather(*tasks)
    
    return results

In [6]:
# ============================================
# CELDA 6 - VARIABLES DE ESTADO
# ============================================

convo_length = 2

unique_conv_id = str(uuid4())

prev_conv = ""

filename = unique_conv_id + "_log.txt"

log_file_path = os.path.join(
    config.LOGS_DIR,
    filename
)

save_file(log_file_path, prev_conv)

primera = True

buscar = False

mi_opcion = None

cat_buscar = 0

graph_data = []

# Estructura:
# 0 - Int_hasCustomer
# 1 - hasSupportCategory
# 2 - hasTypeInc
# 3 - incident_hasOrigin
# 4 - hasSupportGroup
# 5 - hasTechnician

mis_datos = [None, None, None, None, None, None]

print("Sistema inicializado")
print("Estado actual:", mis_datos)


def elemento_mas_comun(array):
    # max() recorre los elementos y los compara basándose en cuántas veces aparecen
    return max(array, key=array.count)

Sistema inicializado
Estado actual: [None, None, None, None, None, None]


In [7]:
# ============================================
# CELDA 7 - FUNCIÓN PRINCIPAL DEL CHAT
# ============================================

def procesar_mensaje_usuario(a):

    global primera
    global mis_datos
    global graph_data
    global cat_buscar
    global mi_opcion
    global prev_conv

    if a == "q":
        print("Finalizando conversación")
        return False

    # ========================================
    # GUARDAR MENSAJE
    # ========================================

    timestamp = time()

    timestring = timestamp_to_datetime(timestamp)

    message = f"USER: {timestring} - {a}"

    # ========================================
    # EXTRACCIÓN DE DATOS
    # ========================================

    if mis_datos[0] is None:

        cliente = extraer_cliente(a)
        print(cliente)

        mis_datos[0] = cliente

    if mis_datos[1] is None:

        support_cat = extraer_support_category(a)

        mis_datos[1] = support_cat

    # ========================================
    # VERIFICAR SI YA ESTÁ COMPLETO
    # ========================================

    if None not in mis_datos and 'None' not in mis_datos:

        print("\nGraphRAG: Query completada")
        print(mis_datos)

        return False

    # ========================================
    # BUSCAR CATEGORÍA FALTANTE
    # ========================================

    try:

        cat_buscar = mis_datos.index(None)

    except ValueError:

        cat_buscar = mis_datos.index('None')

    # ========================================
    # CONSULTA AL GRAFO
    # ========================================
    
    
    #print(mis_datos)
    #print(cat_buscar)
    
    graph_data = buscar_frecuentes_por_opcion(
        graph,
        mis_datos,
        cat_buscar
    )
    
    #print(graph_data)
    
    if graph_data is None or graph_data == []:
        
        graph_data = inferir_valor_adecuado(
            graph,
            mis_datos,
            cat_buscar
        )
    
    #print(graph_data)
    
    # ========================================
    # PREPARACIÓN RESPUESTA
    # ========================================
    
    
    #TODO si falla porque no hay nada ojito
    
    

    max_a_probar = len(graph_data)

    siguiente_a_probar = 1

    retry = True

    while retry:

        retry = False

        prev_conv = open_file(log_file_path)

        if graph_data is None or graph_data == []:

            data = (
                "No se encontraron datos. "
                "Pregunta si el grupo es correcto."
            )

        else:
            mi_opcion = graph_data[0]
            
            data = (
                f"El campo a rellenar es "
                f"{config.DICCIONARIO_PREFIJOS[cat_buscar]}"
                f" y estas son las opciones:\n\nrepcon:"
                f"{config.DICCIONARIO_PREFIJOS[cat_buscar]}"
                f" repcon:"
                f"{mi_opcion}"
            )

        reglas = formatear_para_llm(
            './textos/reglas_incidentes.json',
            tipo_cond=config.DICCIONARIO_PREDICADOS[cat_buscar]
        )

        prompt = open_file(
            config.CONTEXTO_FILE_PATH
        )
        
        
        mensajeinstruc = "Se espera que extraigas el campo " + config.DICCIONARIO_PREFIJOS[cat_buscar]
        
        datos_existentes = formatear_datos_existentes_LLM(graph_data)
        
        print(data)
        
        prompt = (
            prompt
            .replace('<<DATOS>>', data)
            .replace('<<CONVERSACIÓN>>', datos_existentes)
            .replace('<<MENSAJE>>', mensajeinstruc) 
            .replace('<<REGLAS>>', "\n".join(reglas))
        )
        
        
        
        #print(prompt)
        
        # ====================================
        # GENERACIÓN LLM
        # ====================================

        output = text_completion(prompt)
        
        
        
        #paralelo
        
        outputs = asyncio.run(text_completion_batch([prompt,prompt,prompt]))
                
        #print("\n========== RESPUESTA LLM ==========")
        #print("bueno 3")
        #print(output)

        # ====================================
        # LIMPIEZA RESPUESTA
        # ====================================

        #output = extraer_respuesta_limpia_llm(arreglar_lista_llm(output)).replace("repcon:", "")

        nuevo = []
        
        
        
        for x in outputs:
            nuevo.append(extraer_respuesta_limpia_llm(arreglar_lista_llm(x)).replace("repcon:", ""))
        output = elemento_mas_comun(nuevo)
        #print("\nDatos extraídos por el LLM:")
        #print(mis_datos_nuevos)
        #print("bueno")
        #print(output)
        
        mis_datos = merge_lista_y_parametro(mis_datos,output)

        #mis_datos = merge_listas_or(
        #    mis_datos,
        #    mis_datos_nuevos
        #)


        print("\nEstado actualizado:")
        print(mis_datos)

        if output == "ERROR": #TODO Este es el punto clave...

            retry = True

            if siguiente_a_probar < max_a_probar:

                mi_opcion = graph_data[siguiente_a_probar]

                siguiente_a_probar += 1

    # ========================================
    # GUARDAR CONVERSACIÓN
    # ========================================

    timestamp = time()

    timestring = timestamp_to_datetime(timestamp)

    messageBot = f"[Asistente]: {timestring} - {output}"

    save_file(
        log_file_path,
        prev_conv + "\n" + message + "\n" + messageBot
    )

    return True

In [8]:
# ============================================
# CELDA 8 - BUCLE INTERACTIVO
# ============================================

print("====================================")
print(" SISTEMA GraphRAG + Ollama INICIADO")
print("====================================")
print("Escribe 'q' para salir")

primero = True


mis_datos = [None, None, None, None, None, None]

while True:
    
    if primero:
        entrada = input("USER: ")
    
    primero = False
    
    continuar = procesar_mensaje_usuario(entrada)
    
    
    
    
    if not continuar:
        break

#     Hola quiero completar una query. Tengo el supportCategory_149769071762302662 y la empresa company__5B5JVGSPI


 SISTEMA GraphRAG + Ollama INICIADO
Escribe 'q' para salir


USER:  Hola quiero completar una query. Tengo el supportCategory_149769071762302662 y la empresa company__5B5JVGSPI


company__5B5JVGSPI
company__5B5JVGSPI
El campo a rellenar es typeIncident y estas son las opciones:

repcon:typeIncident repcon:typeIncident__1
bueno 3
repcon:typeIncident__4
bueno
typeIncident__1

Estado actualizado:
['company__5B5JVGSPI', 'supportCategory_149769071762302662', 'typeIncident__1', None, None, None]
El campo a rellenar es incidentOrigin y estas son las opciones:

repcon:incidentOrigin repcon:incidentOrigin__2
bueno 3
repcon:incidentOrigin__4
bueno
incidentOrigin__2

Estado actualizado:
['company__5B5JVGSPI', 'supportCategory_149769071762302662', 'typeIncident__1', 'incidentOrigin__2', None, None]
El campo a rellenar es supportGroup y estas son las opciones:

repcon:supportGroup repcon:supportGroup_149763041031762302990
bueno 3
repcon:incidentOrigin__3
bueno
supportGroup__149763041031762302990

Estado actualizado:
['company__5B5JVGSPI', 'supportCategory_149769071762302662', 'typeIncident__1', 'incidentOrigin__2', 'supportGroup__149763041031762302990', None]
El campo a rel

In [9]:
def testear_grafo(g, prefix_uri="http://repcon.org/schema#"):
    """
    Función de diagnóstico para comprobar el estado del grafo y 
    probar las funciones de búsqueda e inferencia de incidentes.
    """
    print("\n" + "="*50)
    print(" INICIANDO TEST DEL GRAFO ".center(50, "="))
    print("="*50)

    # ========================================
    # 1. TAMAÑO DEL GRAFO
    # ========================================
    print("\n[1] Comprobando tamaño del grafo...")
    try:
        print(f"Total de tripletas cargadas: {len(g)}")
    except Exception as e:
        print(f"Error al leer la longitud del grafo: {e}")

   # ========================================
    # 2. TEST DE CONTENIDO BÁSICO (Top 5 Clientes)
    # ========================================
    print("\n[2] Obteniendo los 5 clientes más frecuentes (int_hasCustomer)...")
    query_basica = f"""
    SELECT ?cliente (COUNT(?cliente) AS ?total)
    WHERE {{
        ?incident <{prefix_uri}int_hasCustomer> ?cliente .
    }}
    GROUP BY ?cliente
    ORDER BY DESC(?total)
    LIMIT 5
    """
    
    try:
        # Convertimos a lista para evitar problemas con el generador de rdflib
        resultados = list(g.query(query_basica))
        
        if not resultados:
            print(f"  ⚠ No hay datos para el predicado '{prefix_uri}int_hasCustomer'.")
            print("  🔍 Inspeccionando los 5 predicados que MÁS se repiten en tu grafo...")
            
            query_rescate = """
            SELECT ?p (COUNT(?p) AS ?total)
            WHERE { ?s ?p ?o . }
            GROUP BY ?p
            ORDER BY DESC(?total)
            LIMIT 5
            """
            resultados_rescate = g.query(query_rescate)
            for r in resultados_rescate:
                print(f"    - {r.p} (Apariciones: {r.total})")
        else:
            for row in resultados:
                val = str(row.cliente).split("#")[-1] if "#" in str(row.cliente) else str(row.cliente).rsplit("/", 1)[-1]
                print(f"  - {val} (Apariciones: {row.total})")
                
    except Exception as e:
        print(f"  Error en consulta básica: {e}")

    # ========================================
    # 3. TEST DE TUS FUNCIONES
    # ========================================
    print("\n[3] Probando tus funciones de filtrado e inferencia...")
    
    # Simulamos el array 'mis_datos' (6 posiciones)
    # Suponemos que ya tenemos el cliente (índice 0), y queremos buscar la Categoría (índice 1)
    # Índices: [Customer, SupportCategory, TypeInc, Origin, SupportGroup, Technician]
    
    # ⚠️ IMPORTANTE: Cambia "Cliente_Prueba" por el nombre de un cliente real de tu grafo para testear
    datos_simulados = ["Cliente_Prueba", None, None, None, None, None]
    categoria_a_buscar = 1  # 1 = hasSupportCategory

    print(f"  Estado simulado (mis_datos): {datos_simulados}")
    print(f"  Índice a buscar: {categoria_a_buscar} (hasSupportCategory)")

    # 3.1 Test: buscar_frecuentes_por_opcion
    print("\n  >>> Ejecutando 'buscar_frecuentes_por_opcion'...")
    try:
        res_busqueda = buscar_frecuentes_por_opcion(g, datos_simulados, categoria_a_buscar, prefix_uri)
        print(f"  Resultado Búsqueda Exacta: {res_busqueda}")
    except Exception as e:
        print(f"  Error en buscar_frecuentes_por_opcion: {e}")

    # 3.2 Test: inferir_valor_adecuado
    print("\n  >>> Ejecutando 'inferir_valor_adecuado' (Fallback)...")
    try:
        res_inferencia = inferir_valor_adecuado(g, datos_simulados, categoria_a_buscar, prefix_uri)
        print(f"  Resultado Inferencia: {res_inferencia}")
    except Exception as e:
        print(f"  Error en inferir_valor_adecuado: {e}")

    print("\n" + "="*50)
    print(" TEST FINALIZADO ".center(50, "="))
    print("="*50 + "\n")

In [10]:
# Asumiendo que tu grafo se llama 'graph' en el entorno global:
# ⚠️ Cambia el "Cliente_Prueba" en la función por un string que sepas que sí existe en tu ontología
testear_grafo(graph)


============ INICIANDO TEST DEL GRAFO ============

[1] Comprobando tamaño del grafo...
Total de tripletas cargadas: 513651

[2] Obteniendo los 5 clientes más frecuentes (int_hasCustomer)...
  - company__5B5JVGSPI (Apariciones: 11337)
  - company__UPFP8CUEG (Apariciones: 3526)
  - ss (Apariciones: 3491)
  - company_149767070781762303534 (Apariciones: 2530)
  - company__5RD32STIN (Apariciones: 2410)

[3] Probando tus funciones de filtrado e inferencia...
  Estado simulado (mis_datos): ['Cliente_Prueba', None, None, None, None, None]
  Índice a buscar: 1 (hasSupportCategory)

  >>> Ejecutando 'buscar_frecuentes_por_opcion'...
  Resultado Búsqueda Exacta: []

  >>> Ejecutando 'inferir_valor_adecuado' (Fallback)...
  Resultado Inferencia: []

================ TEST FINALIZADO =================



In [11]:
# ============================================
# CELDA 9 - VISUALIZAR ESTADO FINAL
# ============================================

print("====================================")
print(" ESTADO FINAL")
print("====================================")

labels = [
    "Cliente",
    "SupportCategory",
    "TypeInc",
    "Origin",
    "SupportGroup",
    "Technician"
]

for i, valor in enumerate(mis_datos):

    print(f"{labels[i]} -> {valor}")

 ESTADO FINAL
Cliente -> company__5B5JVGSPI
SupportCategory -> supportCategory_149769071762302662
TypeInc -> typeIncident__1
Origin -> incidentOrigin__2
SupportGroup -> supportGroup__149763041031762302990
Technician -> employee__429


In [12]:
# ============================================
# CELDA 10 - VISUALIZAR LOG
# ============================================

print("====================================")
print(" LOG DE CONVERSACIÓN")
print("====================================")

contenido_log = open_file(log_file_path)

print(contenido_log)

 LOG DE CONVERSACIÓN

USER: Monday, June 08, 2026 at 08:50AM  - Hola quiero completar una query. Tengo el supportCategory_149769071762302662 y la empresa company__5B5JVGSPI
[Asistente]: Monday, June 08, 2026 at 08:50AM  - typeIncident__1
USER: Monday, June 08, 2026 at 08:50AM  - Hola quiero completar una query. Tengo el supportCategory_149769071762302662 y la empresa company__5B5JVGSPI
[Asistente]: Monday, June 08, 2026 at 08:50AM  - incidentOrigin__2
USER: Monday, June 08, 2026 at 08:50AM  - Hola quiero completar una query. Tengo el supportCategory_149769071762302662 y la empresa company__5B5JVGSPI
[Asistente]: Monday, June 08, 2026 at 08:50AM  - supportGroup__149763041031762302990
USER: Monday, June 08, 2026 at 08:50AM  - Hola quiero completar una query. Tengo el supportCategory_149769071762302662 y la empresa company__5B5JVGSPI
[Asistente]: Monday, June 08, 2026 at 08:50AM  - employee__429


In [13]:
import unittest

mis_datos = [None, None, None, None, None, None]

def procesar_query(texto):
    
    veces = 0
    primero = True

    while True:

        if primero:
            entrada = texto

        primero = False

        continuar = procesar_mensaje_usuario(entrada)
        
        if veces >= 10:
            continuar = False
        
        veces = veces +1
        if not continuar:
            break
    
    # Aquí iría tu código real (regex, NLP, etc.)
    # Devuelvo una lista vacía solo para que el código no dé error de compilación
    return mis_datos

class TestQueryProcessor(unittest.TestCase):
    def setUp(self):
        self.casos_de_prueba = [
            ("Hola quiero completar una query. Tengo el supportCategory_149769071762302662 y la empresa company__5B5JVGSPI", 
             ['company__5B5JVGSPI', 'supportCategory_149769071762302662', 'typeIncident__1', 'incidentOrigin__2', 'supportGroup_14976631762302662', 'employee__366']),
            ("Hola quiero completar una query. Tengo el supportCategory_149763371762302662 y la empresa company__O3WHDQU0N", 
             ['company__O3WHDQU0N', 'supportCategory_149763371762302662', 'typeIncident__1', 'incidentOrigin__2', 'supportGroup_14976631762302662', 'employee__366']),
            ("Hola quiero completar una query. Tengo el supportCategory_14976411762302662 y la empresa company__XPERITS62V", 
             ['company__XPERITS62V', 'supportCategory_14976411762302662', 'typeIncident__2', 'incidentOrigin__2', 'supportGroup_14976631762302662', 'employee__294']),
            ("Hola quiero completar una query. Tengo el supportCategory_149769391762302662 y la empresa company__CT7XEFYZJV2", 
             ['company__CT7XEFYZJV2', 'supportCategory_149769391762302662', 'typeIncident__2', 'incidentOrigin__2', 'supportGroup_14976631762302662', 'employee__266']),
            ("Hola quiero completar una query. Tengo el supportCategory_149769771762302662 y la empresa company__O3WHDQU0N", 
             ['company__O3WHDQU0N', 'supportCategory_149769771762302662', 'typeIncident__1', 'incidentOrigin__2', 'supportGroup_14976631762302662', 'employee__366']),
            ("Hola quiero completar una query. Tengo el supportCategory_1497611271762302662 y la empresa company__UPFP8CUEG", 
             ['company__UPFP8CUEG', 'supportCategory_1497611271762302662', 'typeIncident__1', 'incidentOrigin__2', 'supportGroup_14976631762302662', 'employee__366']),
            ("Hola quiero completar una query. Tengo el supportCategory_149761111762302662 y la empresa company__UPFP8CUEG", 
             ['company__UPFP8CUEG', 'supportCategory_149761111762302662', 'typeIncident__1', 'incidentOrigin__2', 'supportGroup_14976631762302662', 'employee__366']),
            ("Hola quiero completar una query. Tengo el supportCategory_149761881762302662 y la empresa company__LABM4K681", 
             ['company__LABM4K681', 'supportCategory_149761881762302662', 'typeIncident__1', 'incidentOrigin__2', 'supportGroup_14976631762302662', 'employee__108']),
            ("Hola quiero completar una query. Tengo el supportCategory_149761881762302662 y la empresa company__09HNKQC8R", 
             ['company__09HNKQC8R', 'supportCategory_149761881762302662', 'typeIncident__1', 'incidentOrigin__2', 'supportGroup_14976631762302662', 'employee__108']),
            ("Hola quiero completar una query. Tengo el supportCategory_14976411762302662 y la empresa company__5B5JVGSPI", 
             ['company__5B5JVGSPI', 'supportCategory_14976411762302662', 'typeIncident__2', 'incidentOrigin__2', 'supportGroup_14976631762302662', 'employee__294']),
            ("Hola quiero completar una query. Tengo el supportCategory_1497661091762302664 y la empresa ss", 
             ['ss', 'supportCategory_1497661091762302664', 'typeIncident__1', 'incidentOrigin__2', 'supportGroup_14976691762302662', 'employee__366'])
        ]

    def test_procesamiento_de_queries(self):
        for input_text, expected_output in self.casos_de_prueba:
            mis_datos[:] = [None, None, None, None, None, None]
            with self.subTest(input_text=input_text):
                resultado = procesar_query(input_text)
                self.assertEqual(resultado, expected_output)


if __name__ == '__main__':
    unittest.main(argv=[''], verbosity=2, exit=False)

test_procesamiento_de_queries (__main__.TestQueryProcessor) ... 

company__5B5JVGSPI
company__5B5JVGSPI
El campo a rellenar es typeIncident y estas son las opciones:

repcon:typeIncident repcon:typeIncident__1
bueno 3
repcon:typeIncident__1
bueno
typeIncident__4

Estado actualizado:
['company__5B5JVGSPI', 'supportCategory_149769071762302662', 'typeIncident__4', None, None, None]
El campo a rellenar es incidentOrigin y estas son las opciones:

repcon:incidentOrigin repcon:incidentOrigin__2
bueno 3
repcon:incidentOrigin__2
bueno
incidentOrigin__2

Estado actualizado:
['company__5B5JVGSPI', 'supportCategory_149769071762302662', 'typeIncident__4', 'incidentOrigin__2', None, None]
El campo a rellenar es supportGroup y estas son las opciones:

repcon:supportGroup repcon:supportGroup_149762921762302662
bueno 3
repcon:supportGroup__14976691762302662
bueno
supportGroup_14976691762302662

Estado actualizado:
['company__5B5JVGSPI', 'supportCategory_149769071762302662', 'typeIncident__4', 'incidentOrigin__2', 'supportGroup_14976691762302662', None]
El campo a re


FAIL: test_procesamiento_de_queries (__main__.TestQueryProcessor) (input_text='Hola quiero completar una query. Tengo el supportCategory_149769071762302662 y la empresa company__5B5JVGSPI')
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/tmp/ipykernel_1312/1148410011.py", line 62, in test_procesamiento_de_queries
    self.assertEqual(resultado, expected_output)
AssertionError: Lists differ: ['com[66 chars]ent__4', 'incidentOrigin__2', 'supportGroup_14[29 chars]429'] != ['com[66 chars]ent__1', 'incidentOrigin__2', 'supportGroup_14[29 chars]366']

First differing element 2:
'typeIncident__4'
'typeIncident__1'

  ['company__5B5JVGSPI',
   'supportCategory_149769071762302662',
-  'typeIncident__4',
?                 ^

+  'typeIncident__1',
?                 ^

   'incidentOrigin__2',
-  'supportGroup_14976691762302662',
?                      ^

+  'supportGroup_14976631762302662',
?                      ^

-  'employee__

bueno 3
repcon:employee__430
bueno
employee__430

Estado actualizado:
['ss', 'supportCategory_1497661091762302664', 'typeIncident__1', 'incidentOrigin__4', 'supportGroup_14976691762302662', 'employee__430']

GraphRAG: Query completada
['ss', 'supportCategory_1497661091762302664', 'typeIncident__1', 'incidentOrigin__4', 'supportGroup_14976691762302662', 'employee__430']
